# 비밀번호 유출 조회의 k-익명성

Pwned Passwords에 SHA-1 해시의 앞 5자리만 보내 유출 횟수를 조회하고, 응답 버킷의 크기와 패딩 효과를 측정합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/hibp_k_anonymity.ipynb)

Colab 기본 CPU 런타임과 Python 표준 라이브러리로 실행합니다. 위에서부터 순서대로 실행하세요.

자기 비밀번호를 넣고 싶으면 마지막 getpass 셀을 쓰세요. 입력한 값은 화면과 노트북에 남기지 않습니다.

## 준비

해시 분리, 범위 조회, 응답 파싱에 공통으로 쓰는 코드를 정의합니다. 조회 데이터는 실행할 때 [Have I Been Pwned의 Pwned Passwords range API](https://haveibeenpwned.com/API/v3#PwnedPasswords)에서 받습니다.

In [ ]:
from dataclasses import dataclass
from getpass import getpass
import hashlib
from http.client import HTTPException
from queue import Empty, Queue
import random
import re
from statistics import median
from threading import Thread
import time
from urllib.request import Request, urlopen


# 공개 예제용 시험값
PASSWORDS = ('password123', 'P@ssw0rd', 'qwerty1234', 'korea1234', 'summer2026!')
RANGE_URL = 'https://api.pwnedpasswords.com/range/'
PREFIX_SPACE = 16 ** 5
SEED = 2026
NETWORK_ERRORS = (OSError, HTTPException)
RESPONSE_LINE = re.compile(r'([0-9A-F]{35}):([0-9]+)')


@dataclass
class Bucket:
    counts: dict
    byte_size: int

    @property
    def size(self):
        return len(self.counts)


class RangeClient:
    def __init__(self, deadline):
        self.deadline = deadline
        self.next_request_at = 0.0

    def fetch(self, prefix, *, padded=False):
        delay = max(0.0, self.next_request_at - time.monotonic())
        if self.deadline - time.monotonic() <= delay:
            raise TimeoutError
        if delay:
            time.sleep(delay)

        headers = {'User-Agent': 'rng-blog-example'}
        if padded:
            headers['Add-Padding'] = 'true'
        request = Request(RANGE_URL + prefix, headers=headers, method='GET')
        timeout = min(8.0, self.deadline - time.monotonic())
        if timeout <= 0:
            raise TimeoutError

        result = Queue(maxsize=1)

        def download():
            try:
                with urlopen(request, timeout=timeout) as response:
                    result.put(response.read())
            except NETWORK_ERRORS as error:
                result.put(error)

        # 소켓 타임아웃과 별도로 DNS와 본문 읽기까지 대기 시간을 제한한다.
        Thread(target=download, daemon=True).start()
        try:
            data = result.get(timeout=timeout)
        except Empty:
            raise TimeoutError from None
        finally:
            self.next_request_at = time.monotonic() + 0.2

        if isinstance(data, NETWORK_ERRORS):
            raise data
        return parse_bucket(data)


def parse_bucket(data):
    counts = {}
    for line in data.decode('ascii').splitlines():
        match = RESPONSE_LINE.fullmatch(line)
        if match is None:
            raise ValueError('응답 줄 형식 오류')
        suffix, count = match.groups()
        if suffix in counts:
            raise ValueError('중복 접미사')
        counts[suffix] = int(count)
    if not counts:
        raise ValueError('빈 응답')
    return Bucket(counts, len(data))


def split_hash(password):
    digest = hashlib.sha1(password.encode('utf-8')).hexdigest().upper()
    return digest, digest[:5], digest[5:]


def verdict(condition):
    return 'PASS' if condition else 'FAIL'


def network_status(error):
    print(f'요청 실패로 건너뜀 ({type(error).__name__})')
    return 'SKIP'


samples = [(password, split_hash(password)) for password in PASSWORDS]
client = RangeClient(deadline=float('inf'))

## 실험 1. SHA-1 해시와 접두사 분리

시험값을 SHA-1으로 해시하고 요청에 보낼 앞 5자리와 로컬에서 비교할 나머지를 확인합니다.

In [ ]:
def experiment_1(samples):
    valid = True
    for password, (digest, prefix, suffix) in samples:
        url = RANGE_URL + prefix
        print(f'시험값 {password}')
        print(f'SHA-1 {digest}')
        print(f'전송 접두 {prefix}')
        print(f'로컬 접미 {suffix}')
        print(url)
        valid &= len(prefix) == 5 and len(suffix) == 35
        valid &= digest == url.removeprefix(RANGE_URL) + suffix
    return verdict(valid)


status_1 = experiment_1(samples)
print(f'판정 {status_1}')

## 실험 2. 시험값 유출 조회

접두사가 같은 해시의 접미사와 횟수를 받아, 시험값의 접미사가 버킷에 있는지 로컬에서 확인합니다. `k`는 응답 버킷의 줄 수입니다.

In [ ]:
def experiment_2(client, samples):
    try:
        for password, (_, prefix, suffix) in samples:
            bucket = client.fetch(prefix)
            count = bucket.counts.get(suffix, 0)
            found = '있음' if suffix in bucket.counts else '없음'
            print(f'{password}  k={bucket.size:,}  일치 {found}')
            print(f'  유출 횟수 {count:,}')
    except NETWORK_ERRORS as error:
        return network_status(error)
    except ValueError:
        print('응답 형식 오류')
        return 'FAIL'
    return 'PASS'


status_2 = experiment_2(client, samples)
print(f'판정 {status_2}')

## 실험 3. 버킷 크기 분포와 패딩

같은 무작위 접두사 표본을 패딩 없이 한 번, `Add-Padding`을 켜고 한 번 조회합니다. 응답 줄 수의 분포와 패딩 응답에서 유출 횟수가 0인 줄의 비율을 비교합니다.

In [ ]:
def print_sizes(label, sizes):
    print(f'{label} 최소 / 중앙값 / 최대 (줄)')
    print(f'  {min(sizes):,} / {median(sizes):,} / {max(sizes):,}')


def experiment_3(client, sample_size):
    sample = random.Random(SEED).sample(range(PREFIX_SPACE), sample_size)
    prefixes = [f'{number:05X}' for number in sample]
    unpadded = []
    print(f'무작위 접두 {sample_size}개 (시드 {SEED})')
    try:
        unpadded = [client.fetch(prefix) for prefix in prefixes]
        print_sizes('패딩 없음', [bucket.size for bucket in unpadded])
        padded = [client.fetch(prefix, padded=True) for prefix in prefixes]
    except NETWORK_ERRORS as error:
        return network_status(error), unpadded
    except ValueError:
        print('응답 형식 오류')
        return 'FAIL', unpadded

    sizes = [bucket.size for bucket in padded]
    print_sizes('패딩 있음', sizes)
    in_range = sum(800 <= size <= 1000 for size in sizes)
    print(f'800~1,000줄인 응답 {in_range}/{sample_size}개')
    zero_count = sum(count == 0 for bucket in padded for count in bucket.counts.values())
    print(f'0회 줄 비율 {zero_count / sum(sizes):.1%}')
    return verdict(in_range == sample_size), unpadded


# 빠르게 확인하려면 표본 크기를 10으로 줄이세요.
sample_size = 50
status_3, unpadded = experiment_3(client, sample_size)
print(f'판정 {status_3}')

## 실험 4. 패딩 없는 응답의 본문 길이

앞에서 받은 패딩 없는 응답 본문의 최소·최대 바이트 수를 비교합니다. 본문 길이가 버킷을 구분하는 단서가 될 수 있는지 확인합니다.

In [ ]:
def experiment_4(unpadded):
    if not unpadded:
        print('패딩 없는 전체 표본이 없어 건너뜀')
        return 'SKIP'
    sizes = [bucket.byte_size for bucket in unpadded]
    smallest, largest = min(sizes), max(sizes)
    print(f'패딩 없는 본문 최소 {smallest:,}바이트')
    print(f'패딩 없는 본문 최대 {largest:,}바이트')
    if smallest < largest:
        print('본문 길이가 버킷마다 달라 단서가 된다.')
    return verdict(smallest < largest)


status_4 = experiment_4(unpadded)
print(f'판정 {status_4}')

## 실험 5. 전체 고유 해시 수 추정

가능한 접두사 수에 패딩 없는 버킷 크기의 표본 중앙값을 곱해 전체 고유 해시 수를 어림합니다.

In [ ]:
def experiment_5(unpadded):
    print(f'접두 공간 16^5 = {PREFIX_SPACE:,}개')
    if not unpadded:
        print('패딩 없는 전체 표본이 없어 건너뜀')
        return 'SKIP'
    typical_size = median(bucket.size for bucket in unpadded)
    estimate = int(PREFIX_SPACE * typical_size)
    print(f'패딩 없는 응답 중앙값 {typical_size:,}줄')
    print(f'전체 고유 해시 수 어림 {estimate:,}개')
    print('표본 중앙값으로 계산한 근사치다.')
    return verdict(estimate > 0)


status_5 = experiment_5(unpadded)
print(f'판정 {status_5}')

## 실험 판정 모아보기

In [ ]:
print('| 실험 | 판정 |')
print('| --- | --- |')
for number, status in enumerate((status_1, status_2, status_3, status_4, status_5), 1):
    print(f'| {number} | {status} |')

## 내 비밀번호 유출 조회 (선택)

준비 셀을 실행한 뒤 이 셀을 실행하면 getpass 입력창이 열립니다. 패딩을 켠 범위 조회로 접두사만 보내고, 일치 여부와 유출 횟수를 표시합니다.

In [ ]:
def check_my_password():
    _, prefix, suffix = split_hash(getpass('비밀번호: '))
    try:
        bucket = client.fetch(prefix, padded=True)
    except (OSError, HTTPException, ValueError):
        raise RuntimeError('조회에 실패했습니다. 잠시 뒤 다시 시도하세요.') from None

    count = bucket.counts.get(suffix, 0)
    print(f"일치 여부 {'있음' if count > 0 else '없음'}")
    print(f'유출 횟수 {count:,}')


check_my_password()